# CheckCIM

In [1]:
import pandas as pd
from sodapy import Socrata
pd.set_option('display.max_rows', 500) 
pd.set_option('display.max_columns', None)
import PySimpleGUI as sg
from pyglet import font
import random
import json
# import OpenGL
# from OpenGL import GLU
font.add_file('/etc/fonts/fonts/CENTAUR.TTF')
cim_url_query = 'data.colorado.gov'
cimDatasets = {}
bicHome = "/home/joe/bic_etl"

def getCimMeta():
    with Socrata(cim_url_query, None) as client:
        datasets = client.datasets()
        for dataset in datasets:
            if dataset['owner']['display_name'] == 'Colorado Information Marketplace':
               title=dataset["resource"]["name"]
               w4x4=dataset["resource"]["id"]
               cimDatasets[title] = dataset
    print(f"CIM Metadata Refreshed with {len(cimDatasets)} found")
    return cimDatasets

cimDatasets = getCimMeta()
print("DONE DONE")

CIM Metadata Refreshed with 598 found
DONE DONE


## Functions

In [2]:
def getCIMInfo(title,datasets):
    print(f'{title}  - {datasets[title]["resource"]["id"]}')
    map = {"object":"Text","int64":"Number","float64":"Number"}
    hist={}
    cimDtypes=[]
    badDtypes=[]

    ngood=0
    nbad=0
    for nn,name in enumerate(datasets[title]["resource"]["columns_name"]):
        cimTyp= datasets[title]["resource"]["columns_datatype"][nn]
      
        tmp=[name,cimTyp]
        cimDtypes.append(tmp)
        if cimTyp not in hist:
            hist[cimTyp]=0
        hist[cimTyp]+=1
        dsc = datasets[title]["resource"]["columns_description"][nn]


    return cimDtypes,hist

#####################################################

def getFile(how,file,WindowP,quoting,header=0):
    if how.lower() == "local":
        print("QUTING 2",quoting)
        if file[-3:].lower() == "tsv":
            try: 
                delim = "\t"
                print("try reading TSV file:",file)
                df=pd.read_csv(file,encoding="latin",delimiter=delim,header=header,quoting=quoting)
          #      headerLength,histNF=analyzeNFields(file,delim=delim)            
            except Exception as err:
                print("failed, now try latin for  reading TSV file:",file)
                
                df=pd.read_csv(file,encoding="latin",delimiter=delim,quoting=quoting)
           #     headerLength,histNF=analyzeNFields(file,delim=delim,encoding="latin")    
        elif file[-4:].lower() == "xlsx":
            df=pd.read_excel(file,engine="openpyxl",header=header)
        elif file[-3:].lower() == "xls":
            df=pd.read_excel(file,header=header)
        else:
            try:
                df=pd.read_csv(file)
         #       headerLength,histNF=analyzeNFields(file)
            except Exception as err:
                print("Error, trying with encoding=latin")
                df=pd.read_csv(file,encoding="latin")
             #   headerLength,histNF=analyzeNFields(file,encoding="latin")
    elif how == "Fetch":
        print("GETTING FETCH FILE")
   #     file=values["-WEB-"]
 #       getPrevFiles(2,file)

        WindowP["-PINFO-"].update(f"START reading WEB File:{file}:")
        df=pd.read_csv(f"{file}?$limit=999999999")
        print("GOT ME A FETCH FILE")
        WindowP["-PINFO-"].update(f"FINISHED reading WEB File:{file}:")
        print("DONE FETCHING ")
    return df

#######################################################

def generate_color_pair():
    """Generates a pair of colors that look good together."""

    # Define some color palettes
    palettes = [
        ["#F8B195", "#F67280", "#C06C84", "#6C5B7B", "#355C7D"],  # Soft palette
        ["#E63946", "#F1FAEE", "#A8DADC", "#457B9D", "#1D3557"],  # Bold palette
        ["#F94144", "#F3722C", "#F8961E", "#F9C74F", "#90BE6D"],  # Vibrant palette
        ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"],  # Earthy palette
    ]

    # Choose a random palette
    palette = random.choice(palettes)

    # Select two random colors from the palette
    color1 = random.choice(palette)
    color2 = random.choice(palette)
    while color1 == color2:  # Ensure they're different
        color2 = random.choice(palette)

    return color1, color2




########################################################
def analyzeData(df,title,datasets):
    print(f'{title}  - {datasets[title]["resource"]["id"]}')
    dfD = df.dtypes.to_dict()
    map = {"object":"Text","int64":"Number","float64":"Number"}
    dfDtypes={}
    hist={}
    cimDtypes=[]
    badDtypes=[]
    cimColumns=[]
    for col,typ in dfD.items():
        dfDtypes[col]={}
        dfDtypes[col]["Pandas"]=str(typ)
        dfDtypes[col]["Mapped"]=map[str(typ)]
    ngood=0
    nbad=0
    notinData={}
    print("------------------------------\nMISMATCHED COLUMNS DATA TYPES")
    try:
        for nn,name in enumerate(datasets[title]["resource"]["columns_name"]):
            cimTyp= datasets[title]["resource"]["columns_datatype"][nn]
            cimColumns.append(name)
            if name in dfDtypes: 
               panTyp = dfDtypes[name]['Mapped']
            else:
               panTyp=""
               notinData[name] = cimTyp
                
            tmp=[name,cimTyp]
            cimDtypes.append(tmp)
            if cimTyp not in hist:
                hist[cimTyp]=0
            hist[cimTyp]+=1
            dsc = datasets[title]["resource"]["columns_description"][nn]
            if cimTyp != panTyp:
                nbad+=1
             #   print(f"{nn}  {name:25.25s}  CIM:{cimTyp:12.12s}  PANDAS:{panTyp}     {dsc}")
                tmp=[name,cimTyp,panTyp,dsc]
                badDtypes.append(tmp)
            else:
                ngood+=1
    except Exception as err:
        print("Roh ROh")
        print(err)
            
    notinCIM={}
    for col in dfDtypes:
        if col not in cimColumns:
            notinCIM[col]=dfDtypes[col]["Mapped"]

    dfMiss = df.isna().sum().to_dict()
    dfNrecs = df.shape[0]
    dfNcols = df.shape[1]
    miss=[]
    for col,cnt in dfMiss.items():
        pp = f"{100*cnt/dfNrecs:6.1f}" 
        if cnt > 0:
            tmp=[col,cnt,dfNrecs,pp]
            miss.append(tmp)

    return badDtypes,miss,[[k,v] for k,v in notinCIM.items()],[[k,v] for k,v in notinData.items()],hist,dfDtypes

########################################################

def showTable(title,info,data,header):

    layout = [
        [sg.Text(title,font='Courier 15 bold ')],
        [sg.Text(info,font='Courier 15 bold ')],
        [sg.Button("Quit")],
        [sg.Table(values=data,
                       vertical_scroll_only=False,col_widths=60,font='Courier 15 bold ' ,
                       auto_size_columns=True,enable_events=True,def_col_width=25,text_color="black",
                       justification='right',
                       key='-TABLE-', headings = header)]
    ]

    window = sg.Window('Table', layout,finalize=True,resizable=True,background_color="#9933ff")

def showTableMulti(title,info,data2,header):
    layouts=[]
    try:
        for nn,data in enumerate(data2):
            print("SHOW ",nn,data)
            print("HEADER ",header[nn])
            print("INFO ",info[nn])
            color_pair = generate_color_pair()

            tmp= [
                  [sg.Text(info[nn],font='Courier 15 bold ')],
                  [sg.Table(values=data,vertical_scroll_only=False,col_widths=60,font='Courier 15 bold ' ,
                           auto_size_columns=True,enable_events=True,def_col_width=25,text_color="black",
                           justification='right',background_color=color_pair[0],alternating_row_color=color_pair[1],
                           key='-TABLE-', headings = header[nn])]
                 ]
            layouts.append(tmp) 
    except Exception as err:
        print("BAd BAD ",err)
        
        
    layout = [
       [sg.Text(title,font='Courier 15 bold ')],
       [sg.Button("Quit")],
       layouts
    ]

    window = sg.Window('Table', layout,finalize=True,resizable=True,background_color="#9933ff")

#############################################################################

def getFileGUI():
    layout = [
    [sg.Button("Quit")],
    [sg.FilesBrowse(button_text="Read File",initial_folder="/home/joe/bic_etl/cdos/business/nonprofit",font="CENTAUR 15",file_types=[("TSV Files","*.tsv"),("CSV Files","*.csv"),("Excel Files","*.xlsx"),("Excel Files","*.xls"),("SIJ Files","*.sij")],enable_events=True,key='-FILE1-')]
    ]

    window = sg.Window('Table', layout,finalize=True,resizable=True,background_color="#cc4400")

###################################################################

def getXrefDict(file):
    xrefs={}
    fin = open(file)
    dct = json.load(fin)
    w4x4 = list(dct.keys())[0]
    for col,dct2 in dct[w4x4].items():
        xrefs[col]=dct2['xref']
    return xrefs

## GUI

In [3]:
titles=sorted(list(cimDatasets.keys()))
layout = [
     [sg.Button('Close',font='Courier 15 bold ')],
     [sg.Text("1. Choose Dataset "),sg.Combo(titles, size=(50,10) , enable_events=True,key='Title',font='Courier 15 bold')],
     [sg.Button('Refresh CIM Meta',key="-refresh-",visible=True)],
#    [sg.Button('Get File',key="getFile",visible=False),
     [sg.FilesBrowse(button_text="Read File",initial_folder="/home/joe/bic_etl/cdos/business/nonprofit",visible=False,font="CENTAUR 15",file_types=[("TSV Files","*.tsv"),("CSV Files","*.csv"),("Excel Files","*.xlsx"),("Excel Files","*.xls"),("SIJ Files","*.sij")],enable_events=True,key='-FILE1-')],
    
       [sg.Radio('No', "-USEXREFS-", default=True, key='-NO-',visible=False),sg.Radio('Yes', "-USEXREFS-",visible=False, key='-YES-')],
     [sg.FilesBrowse(button_text="Get Xref Dict",visible=False,initial_folder="/home/joe/bic_etl/cdos/business/nonprofit/defs",font="CENTAUR 15",file_types=[("JSON Files","*.json"),("TSV Files","*.tsv"),("CSV Files","*.csv"),("Excel Files","*.xlsx"),("Excel Files","*.xls"),("SIJ Files","*.sij")],enable_events=True,key='-GETXREF-')]

     # [sg.Text("Dataset Info: "),sg.Text("",key="-DINFO-")],
     # [sg.Text("2. Get Source Definition File "),sg.FilesBrowse(button_text="Source Def File",target="-SFILE-",initial_folder="/home/joe/bic_etl/cdos/business/nonprofit/defs",font="CENTAUR 15",file_types=[("All Files","*"),("CSV Files","*.csv"),("TSV Files","*.tsv"),("Excel Files","*.xlsx")],enable_events=True,key='-SOURCEFILE-')],
     # [sg.Input("",visible=False,key="-SFILE-",enable_events=True)],
     # [sg.Text("3. Create Source-Transform Dictionary"),sg.Button('Create',font='Courier 15 bold ')],
     # [sg.Text("Input File "),sg.FilesBrowse(button_text="Get File",target="-GINFO-",initial_folder="/home/joe/work",font="CENTAUR 15",file_types=[("All Files","*"),("CSV Files","*.csv"),("TSV Files","*.tsv"),("Excel Files","*.xlsx")],enable_events=True,key='-INPUTFILE-')],
     # [sg.Input("",key="-GINFO-",enable_events=True)],
     # [sg.Button('Show Field Xrefs')]
        
]
window2 = sg.Window('GUI', layout,finalize=True,resizable=True,background_color="#cc4400")
quoting=0
df="" 
xrefsCols=[]
try: 
    while True:
        wid, event, values = sg.read_all_windows()
        print("EV ",event)
        print(values)
        if event == sg.WIN_CLOSED or event == 'Close':
            window2.close()
            break
        elif event == 'Quit':
            wid.close()
        elif event == "Title": 
            title=values["Title"]
            print("Title: ",title)
            cimDtypes,hist=getCIMInfo(title,cimDatasets)
            print(cimDtypes)
            showTable(title,"CIM COLUMN TYPES",cimDtypes,["Column","Data Type"])
  #          showTable(badDtypes,["Column","CIM Type","Panda Type","Description"])
      #      window2["getFile"].update(visible=True) 
            window2["-FILE1-"].update(visible=True) 
            
            window2["-GETXREF-"].update(visible=True) 
            window2["-YES-"].update(visible=True) 
            window2["-NO-"].update(visible=True) 
            
            
            
        elif event == "getFile":
            getFileGUI()
        elif event == "-refresh-":
            cimDatasets = getCimMeta()
        elif event == "-FILE1-":
            file=values["-FILE1-"]
       #     df = pd.read_csv("/home/joe/bic_etl/cdos/business/nonprofit/data_transformed/offices.tsv",delimiter="\t")
            try:
               df = getFile("Local",file,window2,quoting)
               print("DF ",df.head())
            except Exception as err:
                print("DDD Error",err)

            print("STARTING ANALYS",values)
                       
            if (values["-NO-"] == True or  ( values["-YES-"] == True and len(xrefsCols) > 0)): 
                print("VVVVVVVVVV ",values,len(xrefsCols),xrefsCols)
                if (values["-YES-"] == True and len(xrefsCols) > 0):
                    dfN = df.rename(columns=xrefsCols)
                    showTable(title,"SOURCE DATA COLUMN CROSS-REFERENCES",[[key,val] for key,val in xrefsCols.items()],["Orignal Name","Xref Name"])
                    print("Renames Columns",dfN.columns)
                else:
                    dfN = df.copy()
                print("ANALYSIS TIME")
                badDtypes,miss,notinCIM,notinData,oncim,ondata=analyzeData(dfN,title,cimDatasets)
                print("BAD Datatype: ",badDtypes)
                infos = ["COLUMN DATA TYPE MISMATCHES",
                         "CIM COLUMNS NOT FOUND IN DATA",
                         "DATA COLUMNS NOT FOUND ON CIM"]
                headers = [
                    ["Column","CIM Type","Panda Type","Description"],
                    ["Column","Data Type"],
                    ["Column","Data Type"]
                    
                ]
                showTableMulti(title,infos,[badDtypes,notinData,notinCIM],headers)
                print("Done with Multi")
                showTable(title,"SOURCE DATA MISSING COUNTS",miss,["Column","Miss Count","Total Count","% Missing"])
            else:
                try: 
                   print("No XREFS")
                   sg.popup("Please Get Dictionary XREFS You Want to Use")
                except Exception as err:
                    print("ERROR ERROR ",err)
        elif event == "-GETXREF-":
            xrefFile=values["-GETXREF-"]
            print("FOUND ME A XREF",xrefFile)
            
            xrefsCols = getXrefDict(xrefFile)
            print("XREFCOLS ",xrefsCols)
            if values["-YES-"] == True and isinstance(df,pd.DataFrame): 
                dfN = df.rename(xrefsCols)
                print("RENAMED ",dfN.columns)
                badDtypes,miss,notinCIM,notinData,oncim,ondata=analyzeData(dfN,title,cimDatasets)
                print("BAD Datatype: ",badDtypes)
                infos = ["COLUMN DATA TYPE MISMATCHES",
                         "CIM COLUMNS NOT FOUND IN DATA",
                         "DATA COLUMNS NOT FOUND ON CIM"]
                headers = [
                    ["Column","CIM Type","Panda Type","Description"],
                    ["Column","Data Type"],
                    ["Column","Data Type"]
                    
                ]
                showTableMulti(title,infos,[badDtypes,notinData,notinCIM],headers)
                print("Done with Multi")
                showTable(title,"SOURCE DATA MISSING COUNTS",miss,["Column","Miss Count","Total Count","% Missing"])
            
            
except Exception as err:
    print("Got me an Error Here")
    print(err)

EV  Title
{'Title': 'Paid Solicitors Disclosed on Charity Registration Forms in Colorado', '-FILE1-': '', '-NO-': True, '-YES-': False, '-GETXREF-': ''}
Title:  Paid Solicitors Disclosed on Charity Registration Forms in Colorado
Paid Solicitors Disclosed on Charity Registration Forms in Colorado  - wwbh-7bpa
[['entityId', 'Text'], ['documentId', 'Text'], ['fein', 'Text'], ['name', 'Text'], ['nameofPS-PFC-CCV', 'Text'], ['title', 'Text'], ['firstName', 'Text'], ['middleName', 'Text'], ['lastName', 'Text'], ['registrantType', 'Text'], ['address', 'Text'], ['city', 'Text'], ['state', 'Text'], ['zipCode', 'Text'], ['mailingAddress', 'Text'], ['mailingCity', 'Text'], ['mailingState', 'Text'], ['mailingZipCode', 'Text'], ['performedAddress', 'Text'], ['performedCity', 'Text'], ['performedState', 'Text'], ['performedZipCode', 'Text'], ['phone', 'Number']]
EV  -FILE1-
{'Title': 'Paid Solicitors Disclosed on Charity Registration Forms in Colorado', '-FILE1-': '/home/joe/bic_etl/cdos/business/no

/tmp/ipykernel_5452/3421494833.py:32: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(file,encoding="latin",delimiter=delim,header=header,quoting=quoting)


Done with Multi
EV  Quit
{'-TABLE-': []}
EV  Quit
{'-TABLE-': []}
EV  Quit
{'-TABLE-': [], '-TABLE-0': [], '-TABLE-1': []}
EV  Close
{'Title': 'Paid Solicitors Disclosed on Charity Registration Forms in Colorado', '-FILE1-': '/home/joe/bic_etl/cdos/business/nonprofit/data_transformed/char_orgs_sol.tsv', '-NO-': True, '-YES-': False, '-GETXREF-': ''}


### Extra

In [ ]:
quoting=0
delim="\t"
header=0
df=pd.read_csv("/home/joe/bic_etl/cdos/business/nonprofit/data_source/char_orgs_ext.tsv",encoding="latin",delimiter=delim,header=header,quoting=quoting)
 

In [ ]:
df

In [ ]:
dct['icqv-mi3c']

In [ ]:


xrefs = getXrefDict("/home/joe/bic_etl/cdos/business/nonprofit/defs/icqv-mi3c_src_trns_xrefs.json")

In [ ]:
xrefs